# Tabular Kaggle Project

Guideline for steps for the Kaggle Tabular Project. You will "turn in" a GitHub repository, modeled after [Project Template](https://github.com/UTA-DataScience/ProjectTempate) on the day of the final, Friday, May 1 at 11 – 1:30 pm. During the final period we will have about 5 minutes to go over your project and your results.

You can find a list of possible Tabular datasets here on [Excel File in Teams](https://mavsuta.sharepoint.com/:x:/r/teams/Course_2262_data_3402_001-tImyQiF6rCJKf/Shared%20Documents/General/TabularDatasets.xlsx?d=w4ae5174d4ac5455aa4a8f03e70918898&csf=1&web=1&e=Lqtpue). You are not limited to these datasets. If you find a Kaggle challenge not listed that you would like to attempt, please check with Dr. Farbin to make sure it is viable. Note that the requirement is that the data you use is tabular, meaning that it can be represented as a table, therefore excluding images, video, audio, and other more raw data formats as well as data that is more structure.  Note that Kaggle hosts datasets without well defined competition associated with them, which will require you to define the problem and assessment metrics. Please select datasets associated with competitions. 

Your first task is to select a challange / dataset. I would like everyone to come to Lecture on Wednesday April 8 with at least one dataset in mind. I will ask students who select datasets not from the provided list to share the links so we can evaluate the dataset in class.

This notebook outlines the steps you should follow. The file(s) in the GitHub repository should contain these steps. Note that you will be only considering classification projects. 

## Define Project

* Provide Project link.
* Short paragraph describing the challenge. 
* Briefly describe the data.


## Project Description
### Project Link: https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents

The US Accidents Dataset is a comprehensive, real-world dataset that captures traffic accidents across the United States over multiple years (2016 - 2023). 

The primary challenge associated with this dataset is to leverage historical accident data to build a machine learning model capable of predicting the severity of an accident before or as it occurs. This involves identifying how different factors such as weather conditions, time of day, and road infrastructure contribute to accident outcomes.

### Data Description
The dataset includes millions of accident records gathered from real-time and historical data sources such as traffic sensors, government reports, and mapping services. Each record represents a single traffic incident and includes a rich set of features (46 in total) that describe the circumstances surrounding the accident. 

These features include spatial data such as latitude and longitude coordinates, temporal information such as start and end times, and seasons. Additionally, the dataset includes specific weather-related factors such as temperature, humidity, visibility, wind speed, and precipitation, which are important for understanding how the environment effects accident risk and severity. There are also several road and infrastructure markers, such as intersections, traffic signals, stop signs, junctions, and road types, that serve to record the physical context of each incident. The target variable, severity, is an ordinal categorical variable with values ranging from 1 to 4, where higher values indicate a greater impact on traffic flow.

The dataset is high-dimensional and includes a mix of numerical, categorical, and sometimes textual features. It also contains missing values and imbalanced class distributions, which introduce additional preprocessing challenges.

## Data Loading and Initial Look

* Load the data. 
* Count the number of rows (data points) and features.
* Any missing values? 
* Make a table, where each row is a feature or collection of features:
    * Is the feature categorical or numerical
    * What values? 
        * e.g. for categorical: "0,1,2"
        * e.g. for numerical specify the range
    * How many missing values
    * Do you see any outliers?
        * Define outlier.
* For classification is there class imbalance?
* What is the target:
    * Classification: how is the target encoded (e.g. 0 and 1)?
    * Regression: what is the range?

In [2]:
import pandas as pd
import numpy as np

# 1. LOAD THE DATA

file_path = r"C:\Users\moham\Downloads\archive\US_Accidents_March23.csv"
df = pd.read_csv(file_path)


# 2. BASIC INFO

print("Shape of dataset (rows, columns):", df.shape)
print("\nNumber of rows:", df.shape[0])
print("Number of features:", df.shape[1])


# 3. MISSING VALUES

missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing (%)": missing_percent
}).sort_values(by="Missing (%)", ascending=False)

print("\nMissing Values Summary:")
print(missing_df.head(20))

# 4. FEATURE ANALYSIS TABLE

summary_rows = []

for col in df.columns:
    col_data = df[col]
    
    # Determine type
    if pd.api.types.is_numeric_dtype(col_data):
        feature_type = "Numerical"
        
        # Range
        min_val = col_data.min()
        max_val = col_data.max()
        values_info = f"{min_val} to {max_val}"
        
        # Outlier detection using IQR
        Q1 = col_data.quantile(0.25)
        Q3 = col_data.quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = col_data[(col_data < lower_bound) | (col_data > upper_bound)]
        outlier_count = outliers.count()
        
    else:
        feature_type = "Categorical"
        
        unique_vals = col_data.dropna().unique()
        unique_sample = unique_vals[:10]  # limit output
        values_info = ", ".join(map(str, unique_sample))
        
        # No standard outlier detection for categorical
        outlier_count = "N/A"
    
    summary_rows.append({
        "Feature": col,
        "Type": feature_type,
        "Values / Range": values_info,
        "Missing Count": col_data.isnull().sum(),
        "Missing (%)": round(col_data.isnull().mean() * 100, 2),
        "Outlier Count": outlier_count
    })

summary_df = pd.DataFrame(summary_rows)


# 5. DISPLAY SUMMARY TABLE

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

print("\nFeature Summary Table:")
print(summary_df)

# 6. DEFINE OUTLIERS

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\moham\\Downloads\\archive\\US_Accidents_March23.csv'

## Data Visualization

* For classification: compare histogram every feature between the classes. Lots of examples of this in class.
* For regression: 
    * Define 2 or more class based on value of the regression target.
        * For example: if regression target is between 0 and 1:
            * 0.0-0.25: Class 1
            * 0.25-0.5: Class 2
            * 0.5-0.75: Class 3
            * 0.75-1.0: Class 4
    * Compare histograms of the features between the classes.
        
* Note that for categorical features, often times the information in the histogram could be better presented in a table.    
* Make comments on what features look most promising for ML task.

## Data Cleaning and Preperation for Machine Learning

* Perform any data cleaning. Be clear what are you doing, for what feature. 
* Determinine if rescaling is important for your Machine Learning model.
    * If so select strategy for each feature.
    * Apply rescaling.
* Visualize the features before and after cleaning and rescaling.
* One-hot encode your categorical features.

## Machine Learning


### Problem Formulation

* Remove unneed columns, for example:
    * duplicated
    * categorical features that were turned into one-hot.
    * features that identify specific rows, like ID number.
    * make sure your target is properly encoded also.
* Split training sample into train, validation, and test sub-samples.

### Train ML Algorithm

* You only need one algorithm to work. You can do more if you like.
* For now, focus on making it work, rather than best result.
* Try to get a non-trivial result.

### Evaluate Performance on Validation Sample

* Compute the usual metric for your ML task.
* Compute the score for the kaggle challenge.

### Apply ML to the challenge test set

* Once trained, apply the ML algorithm the the test dataset and generate the submission file.
